<div style="text-align:center; border-radius:15px; padding:15px; color:white; margin:0; font-family: 'Orbitron', sans-serif; background: #2E0249; background: #11001C; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.3); overflow:hidden; margin-bottom: 1em;">
  <div style="font-size:150%; color:#FEE100"><b>Global Demanding Jobs Data Analysis and Prediction</b></div>
  <div>This notebook was created with the help of <a href="https://devra.ai/ref/kaggle" style="color:#6666FF">Devra AI</a></div>
</div>

The dataset across the globe is filled with demanding jobs and intriguing nuances that may reveal how industry, region, and experience intersect with salary. If you find these insights useful, please consider upvoting this notebook.

## Table of Contents

- [Imports and Setup](#Imports-and-Setup)
- [Data Loading](#Data-Loading)
- [Data Cleaning and Preprocessing](#Data-Cleaning-and-Preprocessing)
- [Exploratory Data Analysis](#Exploratory-Data-Analysis)
- [Feature Engineering and Modeling](#Feature-Engineering-and-Modeling)
- [Model Evaluation and Predictor](#Model-Evaluation-and-Predictor)
- [Conclusion](#Conclusion)

In [ ]:
# Imports and Setup
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib
matplotlib.use('Agg')  # ensure backend is set before importing pyplot
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.inspection import permutation_importance

# Setting seaborn style for better visuals
sns.set(style='whitegrid', context='notebook')

# A little dry humor: if data were any more demanding, it would ask for a raise.

## Data Loading

In this section, we load our dataset that contains global demanding job information, including job titles, industry, geographic details, experience levels, demand levels, and average salaries in USD.

In [ ]:
# Load the dataset
df = pd.read_csv('global_demanding_jobs_dataset.csv', encoding='ascii', delimiter=',')

# Display first few rows of the data
print('Data Snapshot:')
print(df.head())

## Data Cleaning and Preprocessing

Let's inspect the data for missing values and data types. Note that if any errors are encountered (for example, encoding issues), it's important to address them at the beginning. Since our dataset description is straightforward, we move on to verifying data integrity and performing preprocessing.

In [ ]:
# Check data information
print('Data Information:')
print(df.info())

# Check for missing values
print('\nMissing Values:')
print(df.isnull().sum())

# Drop duplicate rows if any
df.drop_duplicates(inplace=True)

# Since the data doesn't include any explicit date columns, we check that the types match our description.
print('\nUnique Experience Levels:', df['Experience_Level'].unique())
print('Unique Demand Levels:', df['Demand_Level'].unique())

## Exploratory Data Analysis

An insightful analysis often starts with visualizations. We'll explore the distribution of average salaries, job counts by industry, and more. These plots can give us a feel for the data before we move on to model training.

In [ ]:
# Histogram of Avg_Salary_USD
plt.figure(figsize=(8, 5))
sns.histplot(df['Avg_Salary_USD'], kde=True, color='skyblue')
plt.title('Distribution of Average Salary (USD)')
plt.xlabel('Average Salary in USD')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('avg_salary_hist.png')
plt.show()

# Bar plot: Count of jobs by Industry
plt.figure(figsize=(10, 6))
sns.countplot(data=df, y='Industry', order=df['Industry'].value_counts().index, palette='viridis')
plt.title('Job Counts by Industry')
plt.xlabel('Count')
plt.ylabel('Industry')
plt.tight_layout()
plt.savefig('jobs_by_industry.png')
plt.show()

# Box plot: Average Salary by Demand Level
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='Demand_Level', y='Avg_Salary_USD', palette='Set2')
plt.title('Average Salary by Demand Level')
plt.xlabel('Demand Level')
plt.ylabel('Average Salary (USD)')
plt.tight_layout()
plt.savefig('salary_by_demand.png')
plt.show()

# Pie chart using countplot for Work Type distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Work_Type', palette='pastel')
plt.title('Distribution of Work Type')
plt.xlabel('Work Type')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('work_type_distribution.png')
plt.show()

## Feature Engineering and Modeling

For predicting the average salary, we convert categorical variables into numeric via one-hot encoding and drop columns that are not useful for prediction (e.g., ID). A Random Forest Regressor is then used to build our model. Note that while accuracy in classification is usually defined differently, here we use the R² score along with the Root Mean Squared Error for performance evaluation.

In [ ]:
# Prepare the data for modeling

# Drop the ID column as it doesn't contribute to prediction
df_model = df.drop(['ID'], axis=1)

# Identify the target and features
target = 'Avg_Salary_USD'
features = [col for col in df_model.columns if col != target]

# One-hot encode the categorical features
df_encoded = pd.get_dummies(df_model, columns=features, drop_first=True)

print('Shape after encoding:', df_encoded.shape)

# Split the data into training and testing sets
X = df_encoded.drop(target, axis=1)
y = df_encoded[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f'R² Score: {r2:.4f}')
print(f'RMSE: {rmse:.2f}')

# Permutation Importance to gauge feature significance
result = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1)
perm_sorted_idx = result.importances_mean.argsort()

plt.figure(figsize=(8, 10))
plt.barh(range(len(perm_sorted_idx)), result.importances_mean[perm_sorted_idx], align='center')
plt.yticks(range(len(perm_sorted_idx)), X_test.columns[perm_sorted_idx])
plt.xlabel('Mean Permutation Importance')
plt.title('Permutation Importance of Features')
plt.tight_layout()
plt.savefig('permutation_importance.png')
plt.show()

## Model Evaluation and Predictor

Below, we encapsulate our model into a predictor function that takes in new data (in a similar pandas DataFrame format) and returns the predicted average salary. This approach allows for quick predictions if similar datasets are used. Note that for further validation, cross-validation can be performed and hyperparameters tuned.

In [ ]:
def predict_average_salary(new_data):
    """
    Predict the average salary given a new data sample.
    The new_data should be a DataFrame with the same features as used in training (excluding 'ID' and 'Avg_Salary_USD').
    """
    # Ensure we drop the ID column if it exists
    if 'ID' in new_data.columns:
        new_data = new_data.drop('ID', axis=1)
    
    # One-hot encode the input data
    new_data_encoded = pd.get_dummies(new_data, drop_first=True)
    
    # Align the new data with the training data columns
    new_data_encoded = new_data_encoded.reindex(columns=X.columns, fill_value=0)
    
    # Predict using the trained model
    prediction = model.predict(new_data_encoded)
    return prediction

# Example usage:
# Create a small sample input. In practice, ensure the sample contains the same features.
sample_input = df.drop(['ID', 'Avg_Salary_USD'], axis=1).iloc[0:1]
predicted_salary = predict_average_salary(sample_input)
print('Predicted Salary for the sample input:', predicted_salary[0])

## Conclusion

This notebook explored the global demanding jobs dataset through extensive visualizations and data preprocessing techniques. We built a Random Forest Regressor to predict average salaries based on various job-related features and evaluated its performance using R² and RMSE. While the approach provides meaningful initial insights and decent performance, future investigations could involve hyperparameter tuning, cross-validation, and exploring alternative algorithms to further improve prediction accuracy.

If you appreciated this deep dive into the data, please consider upvoting the notebook.